In [3]:
import numpy as np
import pandas as pd

In [4]:
df = pd.read_csv(r"D:\Python\Numpy&Panda\Analysis\sales_data.csv")

Q1. Display:

  the number of rows and columns
  the data types of each column
  any missing values per column

In [5]:
print(f"The number of rows: {df.shape[0]}, columns: {df.shape[1]}")
print(f"The datatype of each column: {df.info()}")
print(f"Missing values: {df.isna().sum()}")

The number of rows: 30, columns: 11
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       30 non-null     int64  
 1   customer_name  30 non-null     object 
 2   age            30 non-null     int64  
 3   city           30 non-null     object 
 4   product        30 non-null     object 
 5   category       30 non-null     object 
 6   quantity       30 non-null     int64  
 7   unit_price     30 non-null     float64
 8   discount_pct   30 non-null     int64  
 9   order_date     30 non-null     object 
 10  status         30 non-null     object 
dtypes: float64(1), int64(4), object(6)
memory usage: 2.7+ KB
The datatype of each column: None
Missing values: order_id         0
customer_name    0
age              0
city             0
product          0
category         0
quantity         0
unit_price       0
discount_pct     0
order_

Q2. Add a new column called revenue which equals:

    quantity * unit_price * (1 - discount_pct / 100)
    Round to 2 decimal places using numpy.
    Then print the min, max, and mean revenue.

In [6]:
df.head()

,order_id,customer_name,age,city,product,category,quantity,unit_price,discount_pct,order_date,status
0,1001,Alice Johnson,28,New York,Laptop,Electronics,1,1200.0,10,2024-01-05,Completed
1,1002,Bob Smith,34,Los Angeles,Headphones,Electronics,2,85.5,5,2024-01-08,Completed
2,1003,Carol White,22,Chicago,Python Book,Books,3,45.0,0,2024-01-10,Completed
3,1004,David Brown,45,Houston,Office Chair,Furniture,1,320.0,15,2024-01-12,Cancelled
4,1005,Eva Martinez,31,Phoenix,Keyboard,Electronics,2,75.0,0,2024-01-15,Completed


In [7]:
df['revenue'] = df['quantity'] * df['unit_price'] * (1 - df['discount_pct'] / 100)
rounded_revenue = np.round(df['revenue'], decimals=2)
print(f"max revenue: {np.max(rounded_revenue)}")
print(f"min revenue: {np.min(rounded_revenue)}")
print(f"mean revenue: {np.mean(rounded_revenue)}")

max revenue: 1215.0
min revenue: 45.0
mean revenue: 376.9566666666667


Q3. Filter the DataFrame to show only:

    - Orders with status == "Completed" AND categroy == "Electronics"
    - From those, find the top 3 orders by revenue (highest first)

In [8]:
filtered = df[
    (df["status"] == "Completed") &
    (df["category"] == "Electronics")
]

filtered

top3 = filtered.sort_values("revenue", ascending=False).head(3)
top3

,order_id,customer_name,age,city,product,category,quantity,unit_price,discount_pct,order_date,status,revenue
11,1012,Liam Thomas,36,Jacksonville,Laptop,Electronics,1,1350.0,10,2024-02-05,Completed,1215.0
0,1001,Alice Johnson,28,New York,Laptop,Electronics,1,1200.0,10,2024-01-05,Completed,1080.0
19,1020,Tina Clark,40,Nashville,Laptop,Electronics,1,999.0,0,2024-03-01,Completed,999.0


Q4. Group by category and compute:

Total revenue

Average discount percentage

Number of orders

Sort results by total revenue descending.

In [9]:
category_stats = (
    df.groupby("category")
      .agg(
          total_revenue=("revenue", "sum"),
          avg_discount=("discount_pct", "mean"),
          order_count=("order_id", "count")
      )
      .sort_values("total_revenue", ascending=False)
)

category_stats

,total_revenue,avg_discount,order_count
category,,,
Electronics,7232.2,7.058824,17
Furniture,3442.0,7.857143,7
Books,634.5,0.833333,6


Q5. Using the order_date column:

    - Extract the month name into a new column called month
    - Find which month had the highest total revenue

In [10]:
df['order_date'] = pd.to_datetime( df['order_date'], errors='coerce' )
df['month'] = df['order_date'].dt.month_name()
print("highest total revenue: \n", df.groupby('month')['revenue'].agg('max') )

highest total revenue: 
 month
February    1215.0
January     1080.0
March        999.0
Name: revenue, dtype: float64


Q6. Using numpy (not pandas built-ins):

    Compute the 25th, 50th, and 75th percentile of unit_price
    Flag orders where revenue is above the 75th percentile as "High Value" and others as "Standard" in a new column order_tier

In [11]:
print("25th: ", np.quantile( df['unit_price'] , 0.25) )
print("50th: ", np.quantile( df['unit_price'] , 0.50) )
print("75th: ", np.quantile( df['unit_price'] , 0.75) )

25th:  62.5
50th:  115.0
75th:  535.0


In [12]:
df['order_tier'] = np.where(df['revenue'] > np.quantile(df['unit_price'], 0.75), "High Value", "Standard")
df.head()

,order_id,customer_name,age,city,product,category,quantity,unit_price,discount_pct,order_date,status,revenue,month,order_tier
0,1001,Alice Johnson,28,New York,Laptop,Electronics,1,1200.0,10,2024-01-05,Completed,1080.00,January,High Value
1,1002,Bob Smith,34,Los Angeles,Headphones,Electronics,2,85.5,5,2024-01-08,Completed,162.45,January,Standard
2,1003,Carol White,22,Chicago,Python Book,Books,3,45.0,0,2024-01-10,Completed,135.00,January,Standard
3,1004,David Brown,45,Houston,Office Chair,Furniture,1,320.0,15,2024-01-12,Cancelled,272.00,January,Standard
4,1005,Eva Martinez,31,Phoenix,Keyboard,Electronics,2,75.0,0,2024-01-15,Completed,150.00,January,Standard


Q7. Create a pivot table showing total revenue for each category × status combination. Fill missing values with 0.

In [13]:
df.pivot_table(index='category', columns='status', values='revenue', aggfunc='sum', fill_value=0)

status,Cancelled,Completed,Pending
category,,,
Books,0.0,634.5,0.0
Electronics,114.0,7118.2,0.0
Furniture,1296.0,624.0,1522.0


Q8. Load the final cleaned DataFrame (with revenue and order_tier) into a database called sales.db, table name orders. Then:

    Query and print the total revenue per category using SQL
    Query and print all cancelled orders

In [14]:
import numpy as np
import pandas as pd
import sqlite3

# Create connection to sales.db (creates file if it doesn't exist)
conn = sqlite3.connect('sales.db')

# Load DataFrame into 'orders' table
# Use if_exists='replace' to overwrite the table if you run this multiple times
df.to_sql('orders', conn, if_exists='replace', index=False)

30

In [15]:
query_revenue = "SELECT category, SUM(revenue) as total_revenue FROM orders GROUP BY category"
revenue_per_cat = pd.read_sql(query_revenue, conn)

print("Total Revenue per Category:")
print(revenue_per_cat)

Total Revenue per Category:
      category  total_revenue
0        Books          634.5
1  Electronics         7232.2
2    Furniture         3442.0


In [16]:
query_cancelled = "SELECT * FROM orders WHERE status = 'Cancelled'"
cancelled_orders = pd.read_sql(query_cancelled, conn)

print("\nCancelled Orders:")
print(cancelled_orders)

# Always close the connection when finished
conn.close()


Cancelled Orders:
   order_id    customer_name  age     city        product     category  \
0      1004      David Brown   45  Houston   Office Chair    Furniture   
1      1011   Karen Anderson   27   Austin         Webcam  Electronics   
2      1018  Rachel Martinez   35  Seattle   Office Chair    Furniture   
3      1027       Aaron King   43   Tucson  Standing Desk    Furniture   

   quantity  unit_price  discount_pct           order_date     status  \
0         1       320.0            15  2024-01-12 00:00:00  Cancelled   
1         1       120.0             5  2024-02-03 00:00:00  Cancelled   
2         1       295.0             0  2024-02-20 00:00:00  Cancelled   
3         1       810.0            10  2024-03-18 00:00:00  Cancelled   

   revenue     month  order_tier  
0    272.0   January    Standard  
1    114.0  February    Standard  
2    295.0  February    Standard  
3    729.0     March  High Value  
